# Agent code

In [0]:
# databricks-vectorsearch>0.73 creates incompatibility with databricks-langchain 0.19.0 when trying to import vectorsearchindex class
%pip install databricks-vectorsearch==0.73
%pip install databricks-langchain
%pip install langmem
%pip install langgraph-checkpoint-postgres

  Attempting uninstall: databricks-vectorsearch
    Found existing installation: databricks-vectorsearch 0.75
    Uninstalling databricks-vectorsearch-0.75:
      Successfully uninstalled databricks-vectorsearch-0.75
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 960.7/960.7 kB 34.7 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
  Attempting uninstall: orjson
    Found existing installation: orjson 3.11.4
    Not uninstalling orjson at /opt/databricks-environments/databricks-ml/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-83136f88-3f5d-468b-9693-2a7f07e43d29
    Can't uninstall 'orjson'. No files were found t

In [0]:
dbutils.library.restartPython()

In [0]:
"""
Imports
"""
from datetime import datetime
from typing import Generator
from typing_extensions import TypedDict
from zoneinfo import ZoneInfo, available_timezones
import mlflow
from databricks_langchain import ChatDatabricks
from langchain_core.messages import AnyMessage, SystemMessage, merge_message_runs, HumanMessage, SystemMessage
from langchain_core.messages.utils import count_tokens_approximately
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.store.base import BaseStore
from langgraph.store.memory import InMemoryStore
from langgraph.store.postgres import PostgresStore
from langgraph.checkpoint.memory import MemorySaver 
from langgraph.prebuilt import ToolNode, tools_condition
from langmem.short_term import RunningSummary, SummarizationNode
from mlflow.entities import SpanType
from mlflow.models import set_model
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (ResponsesAgentRequest
                                    , ResponsesAgentResponse
                                    , ResponsesAgentStreamEvent
                                    , output_to_responses_items_stream
                                    , to_chat_completions_input)
# utility libs
from pydantic import BaseModel, Field
from trustcall import create_extractor
from typing import TypedDict, Literal, Optional
from langchain_openai import ChatOpenAI
from IPython.display import Image, display       
import uuid          

2026-07-01 06:54:31.976040: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-01 06:54:38.893504: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-01 06:54:44.177714: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [0]:
# Utilities
# Inspect the tool calls made by Trustcall
class Spy:
    def __init__(self):
        self.called_tools = []
    def __call__(self, run):
        # Collect information about the tool calls made by the extractor.
        q = [run]
        while q:
            r = q.pop()
            if r.child_runs:
                q.extend(r.child_runs)
            if r.run_type == "chat_model":
                self.called_tools.append(r.outputs["generations"][0][0]["message"]["kwargs"]["tool_calls"])
def extract_tool_info(tool_calls, schema_name="Memory"):
    """Extract information from tool calls for both patches and new memories.
    
    Args:
        tool_calls: List of tool calls from the model
        schema_name: Name of the schema tool (e.g., "Memory", "ToDo", "Profile")
    """

    # Initialize list of changes
    changes = []
    
    for call_group in tool_calls:
        for call in call_group:
            if call['name'] == 'PatchDoc':
                changes.append({
                    'type': 'update',
                    'doc_id': call['args']['json_doc_id'],
                    'planned_edits': call['args']['planned_edits'],
                    'value': call['args']['patches'][0]['value']
                })
            elif call['name'] == schema_name:
                changes.append({
                    'type': 'new',
                    'value': call['args']
                })

    # Format results as a single string
    result_parts = []
    for change in changes:
        if change['type'] == 'update':
            result_parts.append(
                f"Document {change['doc_id']} updated:\n"
                f"Plan: {change['planned_edits']}\n"
                f"Added content: {change['value']}"
            )
        else:
            result_parts.append(
                f"New {schema_name} created:\n"
                f"Content: {change['value']}"
            )
    
    return "\n\n".join(result_parts)
import uuid, psycopg
from databricks.sdk import WorkspaceClient
from langgraph.store.postgres import PostgresStore
def lakebase_conn(instance="lakebase_instance_name"):
    w = WorkspaceClient()
    inst = w.database.get_database_instance(name=instance)
    cred = w.database.generate_database_credential(
        instance_names=[instance], request_id=str(uuid.uuid4()))
    return psycopg.connect(
        host=inst.read_write_dns,
        dbname="databricks_postgres",
        user=w.current_user.me().user_name,
        password=cred.token,
        sslmode="require",
        autocommit=True,
    )

In [0]:
# Prompts
# Chatbot instruction for choosing what to update and what tools to call 

MODEL_SYSTEM_MESSAGE = """You are a helpful chatbot. 

You are designed to be a companion to a user, helping them keep track of their ToDo list.

You have a tool called get_current_time, use this tool if the user asks for the current time and do not use system time

You have a long term memory which keeps track of three things:
1. The user's profile (general information about them) 
2. The user's ToDo list
3. General instructions for updating the ToDo list

Here is the current User Profile (may be empty if no information has been collected yet):
<user_profile>
{user_profile}
</user_profile>

Here is the current ToDo List (may be empty if no tasks have been added yet):
<todo>
{todo}
</todo>

Here are the current user-specified preferences for updating the ToDo list (may be empty if no preferences have been specified yet):
<instructions>
{instructions}
</instructions>

Here are your instructions for reasoning about the user's messages:

1. Reason carefully about the user's messages as presented below. 

2. Decide whether any of the your long-term memory should be updated:
- If personal information was provided about the user, update the user's profile by calling UpdateMemory tool with type `user`
- If tasks are mentioned, update the ToDo list by calling UpdateMemory tool with type `todo`
- If the user has specified preferences for how to update the ToDo list, update the instructions by calling UpdateMemory tool with type `instructions`

3. Tell the user that you have updated your memory, if appropriate:
- Do not tell the user you have updated the user's profile
- Tell the user them when you update the todo list
- Do not tell the user that you have updated instructions

4. Err on the side of updating the todo list. No need to ask for explicit permission.

5. Respond naturally to user user after a tool call was made to save memories, or if no tool call was made."""
# Trustcall instruction
# TRUSTCALL_INSTRUCTION =\
# """Reflect on following interaction. 

# Use the provided tools to retain any necessary memories about the user. 

# Use parallel tool calling to handle updates and insertions simultaneously.

# System Time: {time}"""
TRUSTCALL_INSTRUCTION =\
"""Reflect on following interaction. 

Use the provided tools to retain any necessary memories about the user. 

If more than one tool call, use parallel tool calling to handle updates and insertions simultaneously.

System Time: {time}"""
# Instructions for updating the ToDo list
CREATE_INSTRUCTIONS =\
"""Reflect on the following interaction.

Based on this interaction, update your instructions for how to update ToDo list items. 

Use any feedback from the user to update how they like to have items added, etc.

Your current instructions are:

<current_instructions>
{current_instructions}
</current_instructions>"""

In [0]:
"""
Definitions:
 - Full conversation state. `context` holds the SummarizationNode's RunningSummary
    so it can be reused/merged across model calls within a run.
 - Single get time tool to start with    
"""
class AgentState(MessagesState):
    context: dict[str, RunningSummary]
    # Input schema seen only by the agent node: it reads the *summarized* messages
    # produced by the summarization node, not the raw history.
class LLMInputState(TypedDict):
    summarized_messages: list[AnyMessage]
    context: dict[str, RunningSummary] 
# User profile schema
class Profile(BaseModel):
    """This is the profile of the user you are chatting with"""
    name: Optional[str] = Field(description="The user's name", default=None)
    location: Optional[str] = Field(description="The user's location", default=None)
    job: Optional[str] = Field(description="The user's job", default=None)
    connections: list[str] = Field(
        description="Personal connection of the user, such as family members, friends, or coworkers",
        default_factory=list
    )
    interests: list[str] = Field(
        description="Interests that the user has", 
        default_factory=list
    )
# ToDo schema
class ToDo(BaseModel):
    task: str = Field(description="The task to be completed.")
    time_to_complete: Optional[int] = Field(description="Estimated time to complete the task (minutes).")
    deadline: Optional[datetime] = Field(
        description="When the task needs to be completed by (if applicable)",
        default=None
    )
    solutions: list[str] = Field(
        description="List of specific, actionable solutions (e.g., specific ideas, service providers, or concrete options relevant to completing the task)",
        min_items=1,
        default_factory=list
    )
    status: Literal["not started", "in progress", "done", "archived"] = Field(
        description="Current status of the task",
        default="not started"
    )
# Update memory tool
class UpdateMemory(TypedDict):
    """ Decision on what memory type to update """
    update_type: Literal['user', 'todo', 'instructions']
#memory_model = ChatOpenAI(model="databricks-gemini-3-5-flash", temperature=0)
memory_model = ChatDatabricks(endpoint = 'databricks-gpt-oss-120b')
# Initialize the spy for visibility into the tool calls made by Trustcall
spy = Spy()
# Create the Trustcall extractor for updating the user profile 
profile_extractor = create_extractor(memory_model
                                     , tools = [Profile]
                                     , tool_choice = "Profile").with_listeners(on_end=spy)

@tool
def get_current_time(timezone: str = "UTC") -> str:
    """Get the current date and time.
    Args:
    timezone: An IANA timezone name such as 'UTC', 'America/New_York', or
    'Australia/Melbourne'. Defaults to 'UTC' if omitted or invalid.
    Returns:
    A human-readable timestamp string including the timezone and weekday.
    """
    if timezone not in available_timezones():
        timezone = "UTC"
    now = datetime.now(ZoneInfo(timezone))
    return now.strftime(f"%Y-%m-%d %H:%M:%S {timezone} (%A)")
def agent_node(state: MessagesState, config: RunnableConfig, store: BaseStore):
#def agent_node(state: LLMInputState) -> dict:
    # """Call the tool-bound LLM on the summarized history."""
    # messages = [SystemMessage(content = sys_prompt)] + state["summarized_messages"]
    # response = llm_with_tools.invoke(messages)
    # return {"messages": [response]}
    """Load memories from the store and use them to personalize the chatbot's response."""
    # Get the user ID from the config
    user_id = config["configurable"]["user_id"]

    # Retrieve profile memory from the store
    namespace = ("profile", user_id)
    memories = store.search(namespace)
    if memories:
        user_profile = memories[0].value
    else:
        user_profile = None

    # Retrieve task memory from the store
    namespace = ("todo", user_id)
    memories = store.search(namespace)
    todo = "\n".join(f"{mem.value}" for mem in memories)

    # Retrieve custom instructions
    namespace = ("instructions", user_id)
    memories = store.search(namespace)
    if memories:
        instructions = memories[0].value
    else:
        instructions = ""
    
    system_msg = MODEL_SYSTEM_MESSAGE.format(user_profile=user_profile, todo=todo, instructions=instructions)

    # Respond using memory as well as the chat history
    response = memory_model.bind_tools([UpdateMemory]).invoke([SystemMessage(content=system_msg)]+state["messages"])

    return {"messages": [response]}
def task_mAIstro(state: MessagesState, config: RunnableConfig, store: BaseStore):
    """Load memories from the store and use them to personalize the chatbot's response."""
    # Get the user ID from the config
    user_id = config["configurable"]["user_id"]

    # Retrieve profile memory from the store
    namespace = ("profile", user_id)
    memories = store.search(namespace)
    if memories:
        user_profile = memories[0].value
    else:
        user_profile = None

    # Retrieve task memory from the store
    namespace = ("todo", user_id)
    memories = store.search(namespace)
    todo = "\n".join(f"{mem.value}" for mem in memories)

    # Retrieve custom instructions
    namespace = ("instructions", user_id)
    memories = store.search(namespace)
    if memories:
        instructions = memories[0].value
    else:
        instructions = ""
    
    system_msg = MODEL_SYSTEM_MESSAGE.format(user_profile=user_profile, todo=todo, instructions=instructions)

    # Respond using memory as well as the chat history
    response = memory_model.bind_tools([UpdateMemory]).invoke([SystemMessage(content=system_msg)]+state["messages"])

    return {"messages": [response]}

def update_profile(state: MessagesState, config: RunnableConfig, store: BaseStore):

    """Reflect on the chat history and update the memory collection."""
    
    # Get the user ID from the config
    user_id = config["configurable"]["user_id"]

    # Define the namespace for the memories
    namespace = ("profile", user_id)

    # Retrieve the most recent memories for context
    existing_items = store.search(namespace)

    # Format the existing memories for the Trustcall extractor
    tool_name = "Profile"
    existing_memories = ([(existing_item.key, tool_name, existing_item.value)
                          for existing_item in existing_items]
                          if existing_items
                          else None
                        )

    # Merge the chat history and the instruction
    TRUSTCALL_INSTRUCTION_FORMATTED=TRUSTCALL_INSTRUCTION.format(time=datetime.now().isoformat())
    updated_messages=list(merge_message_runs(messages=[SystemMessage(content=TRUSTCALL_INSTRUCTION_FORMATTED)] + state["messages"][:-1]))

    # Invoke the extractor
    result = profile_extractor.invoke({"messages": updated_messages, 
                                         "existing": existing_memories})
    for r, rmeta in zip(result["responses"], result["response_metadata"]):
        store.put(namespace,
                  rmeta.get("json_doc_id", str(uuid.uuid4())),
                  r.model_dump(mode="json"),
            )
    tool_calls = state['messages'][-1].tool_calls
    return {"messages": [{"role": "tool", "content": "updated profile", "tool_call_id":tool_calls[0]['id']}]}

def update_todos(state: MessagesState, config: RunnableConfig, store: BaseStore):

    """Reflect on the chat history and update the memory collection."""
    
    # Get the user ID from the config
    user_id = config["configurable"]["user_id"]

    # Define the namespace for the memories
    namespace = ("todo", user_id)

    # Retrieve the most recent memories for context
    existing_items = store.search(namespace)

    # Format the existing memories for the Trustcall extractor
    tool_name = "ToDo"
    existing_memories = ([(existing_item.key, tool_name, existing_item.value)
                          for existing_item in existing_items]
                          if existing_items
                          else None
                        )

    # Merge the chat history and the instruction
    TRUSTCALL_INSTRUCTION_FORMATTED=TRUSTCALL_INSTRUCTION.format(time=datetime.now().isoformat())
    updated_messages=list(merge_message_runs(messages=[SystemMessage(content=TRUSTCALL_INSTRUCTION_FORMATTED)] + state["messages"][:-1]))
    
    # Create the Trustcall extractor for updating the ToDo list 
    todo_extractor = create_extractor(memory_model
                                      , tools=[ToDo]
                                      , tool_choice=tool_name
                                      , enable_inserts=True).with_listeners(on_end=spy)

    # Invoke the extractor
    result = todo_extractor.invoke({"messages": updated_messages, 
                                    "existing": existing_memories})

    # Save the memories from Trustcall to the store
    for r, rmeta in zip(result["responses"], result["response_metadata"]):
        store.put(namespace,
                  rmeta.get("json_doc_id", str(uuid.uuid4())),
                  r.model_dump(mode="json"),
            )
        
    # Respond to the tool call made in task_mAIstro, confirming the update
    tool_calls = state['messages'][-1].tool_calls

    # Extract the changes made by Trustcall and add the the ToolMessage returned to task_mAIstro
    todo_update_msg = extract_tool_info(spy.called_tools, tool_name)
    return {"messages": [{"role": "tool", "content": todo_update_msg, "tool_call_id":tool_calls[0]['id']}]}

def update_instructions(state: MessagesState, config: RunnableConfig, store: BaseStore):

    """Reflect on the chat history and update the memory collection."""
    
    # Get the user ID from the config
    user_id = config["configurable"]["user_id"]
    
    namespace = ("instructions", user_id)

    existing_memory = store.get(namespace, "user_instructions")
        
    # Format the memory in the system prompt
    system_msg = CREATE_INSTRUCTIONS.format(current_instructions=existing_memory.value if existing_memory else None)
    new_memory = memory_model.invoke([SystemMessage(content=system_msg)]+state['messages'][:-1] + [HumanMessage(content="Please update the instructions based on the conversation")])

    # Overwrite the existing memory in the store 
    key = "user_instructions"
    store.put(namespace, key, {"memory": new_memory.content})
    tool_calls = state['messages'][-1].tool_calls
    return {"messages": [{"role": "tool", "content": "updated instructions", "tool_call_id":tool_calls[0]['id']}]}
# Conditional edge
def route_message(state: MessagesState, config: RunnableConfig, store: BaseStore) -> Literal[END, "update_todos", "update_instructions", "update_profile", "tools"]:

    """Reflect on the memories and chat history to decide whether to update the memory collection."""
    message = state['messages'][-1]
    if len(message.tool_calls) == 0:
        return END
    else:
        tool_call = message.tool_calls[0]
        if 'update_type' in tool_call['args']:
            if tool_call['args']['update_type'] == "user":
                return "update_profile"
            elif tool_call['args']['update_type'] == "todo":
                return "update_todos"
            elif tool_call['args']['update_type'] == "instructions":
                return "update_instructions"
            else:
                raise ValueError
        else:
            # Regular tool call (e.g. get_current_time) — route to the tools node
            return "tools"
class LangGraphResponsesAgent(ResponsesAgent):
    """Adapter between a compiled LangGraph graph and Databricks Model Serving.
    `to_chat_completions_input` converts the incoming Responses-API messages
    into the chat-completions shape LangGraph expects, and
    `output_to_responses_items_stream` converts each emitted LangGraph message
    (including intermediate tool calls and tool outputs) back into Responses-API
    stream events.
    """
    def __init__(self, agent):
        self.agent = agent
    @mlflow.trace(span_type=SpanType.AGENT)
    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs =\
            [event.item
             for event in self.predict_stream(request)
             if event.type == 'response.output_item.done']
        return ResponsesAgentResponse(output           = outputs
                                    , custom_outputs = request.custom_inputs)
    @mlflow.trace(span_type=SpanType.AGENT)
    def predict_stream(self, request: ResponsesAgentRequest) -> Generator[ResponsesAgentStreamEvent, None, None]:
        cc_msgs = to_chat_completions_input([i.model_dump() for i in request.input])
        #config = {"configurable": {"thread_id": "1", "user_id": "Lance"}}
        if request.custom_inputs:
            config = request.custom_inputs
            agent_stream_generator = self.agent.stream({'messages': cc_msgs}, config, stream_mode=['updates'])
        else:
            agent_stream_generator = self.agent.stream({'messages': cc_msgs}, stream_mode=['updates'])
        for _, events in agent_stream_generator:
            for node_data in events.values():
                # Skip None — LangGraph emits None for nodes that return no state update (e.g. summarize).
                if node_data is None:
                    continue
                # The summarize node emits `summarized_messages`, not `messages`;
                # only agent/tool updates carry messages to surface to the caller.
                messages = node_data.get('messages')
                if messages:
                    yield from output_to_responses_items_stream(messages)        

/home/spark-83136f88-3f5d-468b-9693-2a/.ipykernel/308/command-5312670623036394-2414650216:36: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  solutions: list[str] = Field(
2026/07/01 06:54:58 WARNING mlflow.pyfunc: You have manually traced predict with @mlflow.trace, but this is unnecessary with ResponsesAgent subclasses. You can remove the @mlflow.trace decorator and it will be automatically traced.
2026/07/01 06:54:58 WARNING mlflow.pyfunc: You have manually traced predict_stream with @mlflow.trace, but this is unnecessary with ResponsesAgent subclasses. You can remove the @mlflow.trace decorator and it will be automatically traced.


In [0]:
"""
Inits
 - Any Databricks Foundation Model / external-model serving endpoint that
 - supports tool calling. Change to taste.
"""
llm_endpoint_nme = 'databricks-gpt-oss-120b'
# sys_prompt =\
#     ('You are a helpful assistant. When the user asks anything that depends on '
#      'the current date or time, call the get_current_time tool rather than '
#      'guessing. Always state the timezone in your answer.')
"""
Build agent
Rather than create_react_agent, we wire the graph by hand for full visibility
and control over the control flow:
START -> summarize -> agent -> (tools_condition)
 | |- tools -> summarize (loop)
 | |- END
This makes every node and edge inspectable/customizable: you can insert
guardrails, routing, human-in-the-loop interrupts, or extra tools without
fighting the prebuilt factory.
"""
# Autolog LangChain/LangGraph calls so traces show up in MLflow / the
# Databricks AI Playground.
mlflow.langchain.autolog()
tools = [get_current_time]
# ---------------------------------------------------------------------------
# Build the LangGraph ReAct agent explicitly
# ---------------------------------------------------------------------------
llm = ChatDatabricks(endpoint = llm_endpoint_nme)
#llm_with_tools = llm.bind_tools(tools)
# Store for long-term (across-thread) memory
across_thread_memory = InMemoryStore()
# across_thread_memory = PostgresStore(lakebase_conn(""))
# across_thread_memory.setup()

#accross_thread_memory.setup()

# Checkpointer for short-term (within-thread) memory
within_thread_memory = MemorySaver()
# Pre-model hook. Before each LLM call it condenses older turns once the running
# token count crosses `max_tokens_before_summary`, writing the trimmed list to
# the default `summarized_messages` key while leaving `messages` intact.
summarization_node = SummarizationNode(token_counter = count_tokens_approximately
                                        , model = llm
                                        , max_tokens = 4096 # target budget for the messages handed to the LLM
                                        , max_tokens_before_summary = 4096 # start summarizing once history exceeds this
                                        , max_summary_tokens = 512 # budget reserved for the generated summary
                                        )                                         
# ToolNode executes whatever tool calls the LLM emitted and appends ToolMessages.
tool_node = ToolNode(tools)
builder = StateGraph(AgentState)
# Adding nodes
builder.add_node('summarize', summarization_node)
builder.add_node(update_todos)
builder.add_node(update_profile)
builder.add_node(update_instructions) 
builder.add_node('agent', agent_node)
builder.add_node('tools', tool_node)
# Adding edges
builder.add_edge(START, 'summarize')
builder.add_edge('summarize', 'agent')
builder.add_conditional_edges('agent', route_message)
builder.add_edge('update_todos', 'agent')
builder.add_edge('update_profile', 'agent')
builder.add_edge('update_instructions', 'agent')
# After a regular tool call, return to the agent so it can incorporate the result
builder.add_edge('tools', 'agent')

# tools_condition routes to "tools" when the last message has tool calls,
# otherwise to END.
#builder.add_conditional_edges('agent', tools_condition, {'tools': 'tools', END: END})
# After tools run, re-summarize before the next model call (mirrors a
# pre_model_hook that fires on every turn).
#builder.add_edge('tools', 'summarize')
graph = builder.compile(checkpointer = within_thread_memory
                        , store      = across_thread_memory)

agent = LangGraphResponsesAgent(graph)
set_model(agent)

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
instances = list(w.database.list_database_instances())
for inst in instances:
    print(inst.name)


In [0]:
config = {"configurable": {"thread_id": "1", "user_id": "Sam"}}

# User input to create a profile memory
input_messages = [HumanMessage(content="My name is Sam, I live in Melbourne, I have a wife called Lilly, I like to play games")]

# Run the graph
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

My name is Sam, I live in Melbourne, I have a wife called Lilly, I like to play games


/local_disk0/.ephemeral_nfs/envs/pythonEnv-83136f88-3f5d-468b-9693-2a7f07e43d29/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='content', input_value=[{'type': 'reasoning', 's..., Sam!"\n\nProceed.'}]}], input_type=list])
  return self.__pydantic_serializer__.to_python(


================================== Ai Message ==================================

[{"type": "reasoning", "summary": [{"type": "summary_text", "text": "We need to process user message.\n\nUser gave personal info: name Sam, location Melbourne, wife named Lilly, likes to play games.\n\nThus we need to update user profile memory.\n\nAccording to instructions: if personal info provided, call UpdateMemory tool with type \"user\". Also we should not tell the user we updated profile. Just store and then respond naturally.\n\nSo we need to call function UpdateMemory with update_type \"user\". Probably the tool will handle storing content? The function signature only takes update_type param, not the data. So maybe the system expects we just indicate type and the system automatically stores the message content? Possibly.\n\nThus we call UpdateMemory with type \"user\". Then after tool call, respond e.g., \"Nice to meet you, Sam!\"\n\nProceed."}]}]
Tool Calls:
  UpdateMemory (call_1d3d89e7-9f13-4e

Trace(trace_id=tr-53f8bb06bf480dc1cdb827f43da10cb0)

In [0]:
config = {"configurable": {"thread_id": "1", "user_id": "Sam"}}

# User input to create a profile memory
input_messages = [HumanMessage(content="What time is it?")]

# Run the graph
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

What time is it?
================================== Ai Message ==================================

[{"type": "reasoning", "summary": [{"type": "summary_text", "text": "User asks for current time. Must use get_current_time tool. The tool is not listed in functions; but description says we have tool called get_current_time. Likely exists. We need to call it."}]}]
Tool Calls:
  get_current_time (call_74ffd05f-6418-4664-9b23-55da3ec81416)
 Call ID: call_74ffd05f-6418-4664-9b23-55da3ec81416
  Args:
================================= Tool Message =================================
Name: get_current_time

2026-07-01 04:40:04 UTC (Wednesday)
================================== Ai Message ==================================

[{"type": "reasoning", "summary": [{"type": "summary_text", "text": "We need to respond with the time. Also follow the guidelines: If we used get_current_time tool, respond with natural answer, st

Trace(trace_id=tr-9890b40d1c0f2813669e5cb5f0adac3d)

In [0]:
config = {"configurable": {"thread_id": "1", "user_id": "Sam"}}

# User input to create a profile memory
input_messages = [HumanMessage(content="How many people live in my house? I like to play the following games: DayZ, Diablo4 and Halo")]

# Run the graph
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

How many people live in my house? I like to play the following games: DayZ, Diablo4 and Halo
================================== Ai Message ==================================

[{"type": "reasoning", "summary": [{"type": "summary_text", "text": "User asks \"How many people live in my house?\" They mentioned they have a wife Lilly. So household includes Sam and Lilly = 2 people. Also they said they like to play games: list includes DayZ, Diablo4, Halo. This is personal info about interests. Should we update profile? The profile already has interests: ['playing games']. Should we update to include specific games? The profile's interests field could be updated, but the instruction says update_profile node if personal information provided. This includes new interests. So we should update user profile with updated interests list adding those games. But we should not inform user about profile update. Also answer 

Trace(trace_id=tr-3c5f253423a1b29e3339d8efd8ebd8a6)

In [0]:
user_id = "Sam"
# Search 
for memory in across_thread_memory.search(('profile', user_id)):
    print(memory.value)

{'name': 'Sam', 'location': 'Melbourne', 'job': None, 'connections': ['Lilly (wife)'], 'interests': ['DayZ', 'Diablo4', 'Halo']}


In [0]:
config = {"configurable": {"thread_id": "1", "user_id": "Sam"}}

# User input to create a profile memory
input_messages = [HumanMessage(content="Does my wife play games? How many houses do I have in Melbourne? I live in the same suburb as my Mother")]

# Run the graph
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

Does my wife play games? How many houses do I have in Melbourne? I live in the same suburb as my Mother
================================== Ai Message ==================================

[{"type": "reasoning", "summary": [{"type": "summary_text", "text": "The user asks three questions:\n\n1. Does my wife play games? No info given, can't assume. Should respond that we don't have that info.\n\n2. How many houses do I have in Melbourne? No info, assume one unless known. We only know they live in Melbourne, not number of houses. Could answer based on typical assumption: likely one. But we don't know. Could ask for clarification? But guidelines: respond naturally. Might say we don't have that info.\n\n3. I live in the same suburb as my Mother. That's new personal info: mother lives in same suburb. Should update profile with mother connection? Currently connections list only includes \"Lilly (wife)\". Should add

Trace(trace_id=tr-34fda6ff4c7935ed3e82ba192b2ff3b3)

In [0]:
user_id = "Sam"
# Search 
for memory in across_thread_memory.search(('profile', user_id)):
    print(memory.value)

{'name': 'Sam', 'location': 'Melbourne', 'job': None, 'connections': ['Lilly (wife)', 'Mother'], 'interests': ['DayZ', 'Diablo4', 'Halo']}


In [0]:
schema_name = "Memory"
changes = extract_tool_info(spy.called_tools, schema_name)
print(changes)

In [0]:
# User input for a ToDo
input_messages = [HumanMessage(content="My wife asked me to book a repairmen to fix the windows on our house, can you add this to my list of todo?")]

# Run the graph
for chunk in graph.stream({"messages": input_messages}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

My wife asked me to book a repairmen to fix the windows on our house, can you add this to my list of todo?
================================== Ai Message ==================================

Okay, I've added that to your to-do list.

```
<todo>
- Book a repairmen to fix the windows on our house
</todo>
```


Trace(trace_id=tr-3528aa3ed8a482643c78467aa8a21ad2)

In [0]:
# Example: Pass a string to your agent for inference
from mlflow.types.responses import ResponsesAgentRequest

config = {"configurable": {"thread_id": "1", "user_id": "Sam"}}
request = ResponsesAgentRequest(input=[{"role": "user", "content": "My name is Sam, I live in Melbourne, I have a wife called Lilly, I like to play games"}], custom_inputs = config)
response = agent.predict(request)

Trace(trace_id=tr-8720c7e7790515b3f48b12faafc1af8c)

In [0]:
config = {"configurable": {"thread_id": "1", "user_id": "Sam"}}
request = ResponsesAgentRequest(input=[{"role": "user", "content": "What time is it?"}], custom_inputs = config)
response = agent.predict(request)

Trace(trace_id=tr-cc1a20d90a6b5e155b56592610982fa5)

In [0]:
# Print all memories stored in across_thread_memory with their namespaces and keys
for memory in across_thread_memory.search('user'):
    print(f"Namespace: {memory.namespace}, Key: {memory.key}, Value: {memory.value}")

In [0]:
https://docs.langchain.com/oss/python/langchain/guardrails